# Bước B4 (Hướng A — Dataset lớn 60.000 ảnh, 5 phiên): `DINOv3` + `NIW` + `FTR1` + `DYDM` trên `CIFAR-100`
- **Nhánh:** `duytung/15-cifar100-dinov3-dydm`
- **Dataset:** `CIFAR-100` (`60.000` ảnh, `100` lớp: `50` lớp gốc `S0` + `5 × 10` lớp mới không nhãn `S1..S5` — tự động tải từ HuggingFace `uoft-cs/cifar100`, **không cần upload Drive!**).
- **Tối ưu tốc độ:** Chạy **Pure Closed-Form `NIW` + `FTR1` + `DYDM`** (bỏ qua nút thắt 15 giờ của SVI JAX) → rút gọn toàn bộ 60.000 ảnh × 5 phiên × 5 seed xuống chỉ còn **~85–95 phút trên GPU L4**!
- **Mốc đối chiếu:** Bài báo gốc `VB-CGCD (ICML 2025)`: `S0_All = 91.68%`, `M_d = 79.14%`, `M_f = 6.94%`, `S5_All = 81.23%`.

In [ ]:
# Cell 1: Kết nối Google Drive & kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
# Cell 2: Clone/pull đúng nhánh duytung/15-cifar100-dinov3-dydm
import os, subprocess

REPO_URL = 'https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git'
BRANCH = 'duytung/15-cifar100-dinov3-dydm'
REPO_DIR = '/content/VB-CGCD-defect-testing-only'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'], check=True)

subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], check=True)

In [ ]:
# Cell 3: Cài đặt thư viện (nếu runtime mới)
!pip install -q peft transformers accelerate scipy scikit-learn datasets psutil prettytable
!python -c "import torch, transformers, peft; print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())"

In [ ]:
# Cell 4: Chạy nền Bước B4 (CIFAR-100 DINOv3 + NIW + FTR1 + DYDM) có auto-resume trên Google Drive
import os, subprocess
from google.colab import userdata

try:
    hf_tok = userdata.get('HF_TOKEN')
except Exception:
    hf_tok = os.environ.get('HF_TOKEN', '')

assert hf_tok, 'Vui lòng cấu hình HF_TOKEN trong Colab Secrets (biểu tượng chìa khóa bên trái)!'

env = os.environ.copy()
env['HF_TOKEN'] = hf_tok
env['HUGGING_FACE_HUB_TOKEN'] = hf_tok

log_path = '/content/drive/MyDrive/VB_CGCD_Work/c100_b4_dinov3_dydm.log'
os.makedirs(os.path.dirname(log_path), exist_ok=True)
pid_path = '/tmp/c100_b4.pid'

with open(log_path, 'a') as lf:
    p = subprocess.Popen(
        ['bash', '/content/VB-CGCD-defect-testing-only/colab/run_c100_b4_bg.sh'],
        stdout=lf, stderr=subprocess.STDOUT, env=env, start_new_session=True
    )
with open(pid_path, 'w') as pf:
    pf.write(str(p.pid))
print(f'Đã khởi chạy Bước B4 (CIFAR-100 DINOv3 + DYDM) chạy nền! PID={p.pid} | Log: {log_path}')

In [ ]:
# Cell 5: Kiểm tra tiến độ & xem bảng tổng hợp khi chạy xong
from pathlib import Path

pid = int(Path('/tmp/c100_b4.pid').read_text().strip()) if Path('/tmp/c100_b4.pid').exists() else -1
alive = False
if pid > 0 and Path(f'/proc/{pid}/status').exists():
    st = Path(f'/proc/{pid}/status').read_text()
    alive = ('State:\tZ' not in st and 'State: Z' not in st)
print(f'Trạng thái PID={pid}:', 'ĐANG CHẠY' if alive else 'ĐÃ DỪNG / XONG')

log_p = Path('/content/drive/MyDrive/VB_CGCD_Work/c100_b4_dinov3_dydm.log')
if log_p.exists():
    lines = log_p.read_text(errors='replace').splitlines()
    print('\n--- 40 dòng log gần nhất ---')
    print('\n'.join(lines[-40:]))

sum_p = Path('/content/drive/MyDrive/VB_CGCD_Work/c100_b4_dinov3_dydm_summary.txt')
if sum_p.exists() and not alive:
    print('\n--- BẢNG TỔNG HỢP BƯỚC B4 (CIFAR-100 DINOv3 + DYDM) ---')
    print(sum_p.read_text())